# **EDA & feature engineering**

In [1]:
import os
import re
import ast # Added for literal_eval
import pandas as pd
import numpy as np
import collections
import plotly.express as px
import plotly.graph_objects as go

In [3]:
DATA_DIR = "/Users/aknur/Downloads/final-project-adml/data"
RESULTS_DIR = "/Users/aknur/Downloads/final-project-adml/results"
FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")
SUMMARY = os.path.join(RESULTS_DIR, "eda_summary.txt")
IN_CSV = os.path.join(DATA_DIR, "preprocessed.csv")

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [4]:
def section(title: str) -> None:
    print(f"\n{'='*60}\n  {title}\n{'='*60}")

def save_fig(filename: str, fig) -> None:
    fig.show()  # Opens the visualization in your default web browser
    filepath = os.path.join(FIGURES_DIR, filename)
    fig.write_html(filepath)
    print(f"  Saved -> {filepath}")

In [5]:
# ── Kazakh stop words (from 04_feature_engineering.py) ────────────────────────
KAZ_STOPWORDS = list({
    "және", "да", "де", "бар", "жоқ", "бұл", "осы", "оның", "ол",
    "олар", "біз", "сіз", "мен", "сен", "үшін", "деп", "болды",
    "болған", "болып", "деген", "туралы", "екен", "еді", "алды",
    "қол", "кейін", "бойынша", "арқылы", "ретінде", "дейін",
    "жылы", "жыл", "ай", "күн", "қаласы", "қаласында", "облысы",
    "облысында", "республикасы", "оны", "оларды", "барлық", "онда",
    "жасалды", "болса", "сондай", "сонымен", "сол", "соның",
    "бірге", "бірақ", "немесе", "не", "яғни", "осыған", "үстінде",
    "астында", "қарай", "қатысты", "хабарлады", "айтты", "мәлімет",
    "хабарлайды", "деді", "берді", "жасады", "атап", "тыс",
})

In [6]:
# ── Tokenizer (inferred from usage) ───────────────────────────────────────────
def tokenise(text: str) -> list[str]:
    if not isinstance(text, str):
        return []
    text = text.lower()
    # Keep only Cyrillic, Latin letters and numbers, split by non-alphanumeric
    tokens = re.findall(r"[а-яёәғіқңөүһa-z0-9]+", text)
    return [t for t in tokens if t not in KAZ_STOPWORDS and len(t) > 1] # Filter short tokens and stopwords


In [8]:
df = pd.read_csv(IN_CSV)
print(f"Loaded total rows : {len(df):,}")

Loaded total rows : 27,743


In [9]:
# LABELED dataframe for category-specific analysis
LABELED = df[df["category"] != "басқа"].copy() # Filter out 'other' category for labeled analysis
LABELED = LABELED[LABELED["text"].notna()].copy()
LABELED = LABELED.reset_index(drop=True)

print(f"Labeled rows for analysis : {len(LABELED):,}")
print(f"Categories   : {LABELED['category'].value_counts().to_dict()}")

Labeled rows for analysis : 27,743
Categories   : {'other': 18046, 'health': 2854, 'politics': 1953, 'sports': 1242, 'crime': 1152, 'international': 912, 'economy': 642, 'education': 528, 'social': 414}


In [10]:
cat_counts = LABELED["category"].value_counts()
PALETTE = px.colors.qualitative.Plotly # Use a plotly palette
lines = [] # For summary text file

In [11]:

# ══════════════════════════════════════════════════════════════
# 2. Text length by category
# ══════════════════════════════════════════════════════════════
section("2. Text length by category")
order = cat_counts.sort_values(ascending=False).index.tolist()
fig = px.box(
    LABELED,
    x="category",
    y="char_count",
    category_orders={"category": order},
    color="category", # To use different colors for each box, similar to seaborn palette
    color_discrete_sequence=PALETTE,
    title="Категория бойынша мәтін ұзындығының таралуы",
    labels={
        "category": "Категория",
        "char_count": "Мәтін ұзындығы (таңба)"
    },
    hover_data={"category": False, "char_count": True} # Show char_count on hover
)
fig.update_traces(boxpoints=False) # Equivalent to showfliers=False
fig.update_layout(xaxis_tickangle=-30) # Rotate x-axis labels
save_fig("eda_02_text_length_by_category.html", fig)

lines.append("=== Text length stats per category ===")
lines.append(
    LABELED.groupby("category")["char_count"]
    .describe()[["mean", "50%", "max"]].round(0).to_string()
)
lines.append("")


  2. Text length by category


  Saved -> /Users/aknur/Downloads/final-project-adml/results/figures/eda_02_text_length_by_category.html


In [12]:
# 3. Word count histogram
# ══════════════════════════════════════════════════════════════
section("3. Word count histogram")

# Clip data for histogram as in original
clipped_word_count = LABELED["word_count"].clip(upper=1000)

fig = px.histogram(
    clipped_word_count,
    nbins=60,
    title="Мақаладағы сөз санының гистограммасы",
    labels={
        "value": "Сөз саны (мақалаға)",
        "count": "Мақала саны"
    },
    color_discrete_sequence=[PALETTE[0]]
)
# Add median line
median_word_count = LABELED["word_count"].median()
fig.add_vline(
    x=median_word_count,
    line_dash="dash",
    line_color="red",
    annotation_text=f'Медиана: {median_word_count:.0f}',
    annotation_position="top right"
)
fig.update_layout(showlegend=False) # Hide legend for the median line if it appears
save_fig("eda_03_word_count_histogram.html", fig)

lines.append("=== Word count statistics ===")
lines.append(LABELED["word_count"].describe().round(1).to_string())
lines.append("")


  3. Word count histogram


  Saved -> /Users/aknur/Downloads/final-project-adml/results/figures/eda_03_word_count_histogram.html


In [13]:

# ══════════════════════════════════════════════════════════════
# 4. Top-30 words overall
# ══════════════════════════════════════════════════════════════
section("4. Top-30 words overall")
all_tokens = []
for text in LABELED["text"].dropna():
    all_tokens.extend(tokenise(text))



counter = collections.Counter(all_tokens)
top30 = counter.most_common(30)
words, freqs = zip(*top30)

lines.append("=== Top 30 words overall ===")
for w, f in top30:
    lines.append(f"  {w}: {f:,}")
lines.append("")

# Create a DataFrame for plotly
df_top_words = pd.DataFrame({"word": words, "frequency": freqs})

fig = px.bar(
    df_top_words,
    x="word",
    y="frequency",
    title="30 ең жиі кездесетін сөздер (стоп-сөздерсіз)",
    labels={
        "word": "Сөз",
        "frequency": "Жиілік"
    },
    color_discrete_sequence=[PALETTE[1]])
fig.update_layout(xaxis_tickangle=-45) # Rotate x-axis labels
save_fig("eda_04_top30_words.html", fig)



  4. Top-30 words overall


  Saved -> /Users/aknur/Downloads/final-project-adml/results/figures/eda_04_top30_words.html


In [14]:

# ══════════════════════════════════════════════════════════════
# 5. Top-10 words per category (small multiples)
# ══════════════════════════════════════════════════════════════
section("5. Top-10 words per category")

categories = order # 'order' is from section 2, sorted category names
lines.append("=== Top 10 words per category ===")

for i, cat in enumerate(categories): # Iterate and create individual plots
    subset = LABELED[LABELED["category"] == cat]["text"].dropna()
    tokens = []
    for t in subset:
        tokens.extend(tokenise(t))
    top10 = collections.Counter(tokens).most_common(10)

    if top10:
        ws, fs = zip(*top10)
        df_cat_words = pd.DataFrame({"word": ws, "frequency": fs})

        fig = px.bar(
            df_cat_words,
            x="frequency",
            y="word",
            orientation="h", # Horizontal bar chart
            title=f"Категория: {cat} - Ең жиі 10 сөз",
            labels={
                "word": "Сөз",
                "frequency": "Жиілік"
            },
            color_discrete_sequence=[PALETTE[i % len(PALETTE)]])
        fig.update_layout(yaxis={'categoryorder':'total ascending'}) # Invert y-axis to show highest at top
        save_fig(f"eda_05_top10_words_category_{cat}.html", fig)

        lines.append(f"\nCategory: {cat}")
        for w, f in top10:
            lines.append(f"  {w}: {f:,}")
    else:
        print(f"  No tokens found for category: {cat}")
lines.append("")



  5. Top-10 words per category


  Saved -> /Users/aknur/Downloads/final-project-adml/results/figures/eda_05_top10_words_category_other.html


  Saved -> /Users/aknur/Downloads/final-project-adml/results/figures/eda_05_top10_words_category_health.html


  Saved -> /Users/aknur/Downloads/final-project-adml/results/figures/eda_05_top10_words_category_politics.html


  Saved -> /Users/aknur/Downloads/final-project-adml/results/figures/eda_05_top10_words_category_sports.html


  Saved -> /Users/aknur/Downloads/final-project-adml/results/figures/eda_05_top10_words_category_crime.html


  Saved -> /Users/aknur/Downloads/final-project-adml/results/figures/eda_05_top10_words_category_international.html


  Saved -> /Users/aknur/Downloads/final-project-adml/results/figures/eda_05_top10_words_category_economy.html


  Saved -> /Users/aknur/Downloads/final-project-adml/results/figures/eda_05_top10_words_category_education.html


  Saved -> /Users/aknur/Downloads/final-project-adml/results/figures/eda_05_top10_words_category_social.html


In [15]:

# ══════════════════════════════════════════════════════════════
# 6. Source breakdown pie
# ══════════════════════════════════════════════════════════════
section("6. Source breakdown")

src = df["source"].value_counts().reset_index()
src.columns = ["source", "count"]
labels_map = {"kaggle": "Kaggle (2020–2023)", "parsed": "Parsed (2024–2026)"}
src["source_label"] = src["source"].map(labels_map)

fig = px.pie(
    src,
    values="count",
    names="source_label",
    title="Деректер көздері",
    color_discrete_sequence=[PALETTE[0], PALETTE[3]] # Match original colors if possible
)
fig.update_traces(textposition="inside", textinfo="percent+label")
save_fig("eda_06_source_breakdown.html", fig)

lines.append("=== Source breakdown ===")
lines.append(df["source"].value_counts().to_string())
lines.append("")



  6. Source breakdown


  Saved -> /Users/aknur/Downloads/final-project-adml/results/figures/eda_06_source_breakdown.html


In [16]:

# ══════════════════════════════════════════════════════════════
# 7. Monthly article volume (parsed subset)
# ══════════════════════════════════════════════════════════════
section("7. Monthly volume – parsed subset")

parsed = df[df["source"] == "parsed"].copy()
parsed = parsed[parsed["date"].notna() & parsed["date"].ne("")]
parsed["date_dt"] = pd.to_datetime(parsed["date"], errors="coerce")
parsed = parsed.dropna(subset=["date_dt"])
parsed["month"] = parsed["date_dt"].dt.to_period("M")

if len(parsed) > 0:
    monthly = parsed.groupby("month").size().reset_index(name="count")
    monthly["month"] = monthly["month"].astype(str) # Convert Period to string for plotting

    fig = px.line(
        monthly,
        x="month",
        y="count",
        title="Айлық мақала динамикасы (parsed деректер)",
        labels={
            "month": "Ай",
            "count": "Мақала саны"
        },
        markers=True,
        color_discrete_sequence=[PALETTE[4]]
    )
    fig.update_traces(fill="tozeroy", opacity=0.2) # Equivalent to fill_between
    fig.update_layout(xaxis_tickangle=-30)
    save_fig("eda_07_monthly_volume.html", fig)
    lines.append("=== Monthly volume (parsed) ===")
    lines.append(monthly.set_index("month")["count"].to_string()) # Convert back to series for to_string
    lines.append("")
else:
    print("  Not enough dated rows – skipping plot 7")



  7. Monthly volume – parsed subset


  Saved -> /Users/aknur/Downloads/final-project-adml/results/figures/eda_07_monthly_volume.html


In [17]:


# ══════════════════════════════════════════════════════════════
# 8. Tag co-occurrence (top 20 pairs)
# ══════════════════════════════════════════════════════════════
section("8. Tag co-occurrence")

# Need to parse tags column for co-occurrence analysis, as it's loaded as string
def parse_tags_from_string(tag_string):
    if pd.isna(tag_string):
        return []
    try:
        parsed = ast.literal_eval(tag_string)
        if isinstance(parsed, list):
            return parsed
        else:
            return []
    except (ValueError, SyntaxError):
        return [t.strip() for t in str(tag_string).split(',') if t.strip()]

LABELED["tags_list_parsed"] = LABELED["tags"].apply(parse_tags_from_string)

pair_counter: collections.Counter = collections.Counter()
for tag_list in LABELED["tags_list_parsed"]: # Use the parsed list
    if isinstance(tag_list, list) and len(tag_list) >= 2:
        for j in range(len(tag_list)):
            for k in range(j + 1, len(tag_list)):
                pair = tuple(sorted([tag_list[j], tag_list[k]]))
                pair_counter[pair] += 1
top_pairs = pair_counter.most_common(20)
if top_pairs:
    pair_labels = [f"{a} ↔ {b}" for (a, b), _ in top_pairs]
    pair_counts = [c for _, c in top_pairs]
    df_cooccurrence = pd.DataFrame({"pair": pair_labels, "count": pair_counts})

    fig = px.bar(df_cooccurrence.sort_values("count", ascending=True), # Sort for horizontal bar chart
                 x="count", y="pair", orientation="h",
                 title="Тег жұптарының ең жиі бірігуі (Top 20)",
                 labels={"pair": "Тег жұбы", "count": "Бірлескен кездесу саны"},
                 color_discrete_sequence=[PALETTE[6]])
    save_fig("eda_08_tag_cooccurrence.html", fig)

    lines.append("=== Top 20 tag co-occurrence pairs ===")
    for (a, b), c in top_pairs:
        lines.append(f"  ({a}, {b}): {c}")
    lines.append("")



  8. Tag co-occurrence


  Saved -> /Users/aknur/Downloads/final-project-adml/results/figures/eda_08_tag_cooccurrence.html


In [18]:

with open(SUMMARY, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
print(f"  Saved → {SUMMARY}")

  Saved → /Users/aknur/Downloads/final-project-adml/results/eda_summary.txt
